# Baseline - Regresión Logística Multiclase y K-NN
---

## Justificación de los modelos seleccionados

La **Regresión Logística** permite analizar si las clases son separables mediante combinaciones lineales de las características. Si su rendimiento es inferior al de modelos no lineales, se confirma la necesidad de fronteras de decisión más complejas.

**K-NN**, por su parte, permite evaluar si las configuraciones manuales similares se agrupan naturalmente en el espacio de características. Este modelo clasifica en función de la proximidad entre muestras, sin asumir una forma paramétrica explícita.

Estas dos aproximaciones se usan aquí como referencias iniciales para comparar posteriormente con modelos más complejos.

In [13]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

In [14]:
import sys
from pathlib import Path

SEED = 42
np.random.seed(SEED)

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.data import DataLoader

Tras definir las librerías necesarias, se cargarán los datos de entrenamiento, validación y prueba mediante `DataLoader`, manteniendo el conjunto de prueba completamente aislado para evaluación final.

In [15]:
loader = DataLoader()
X_train, y_train, X_val, y_val, X_test, y_test = loader.load_data()
class_names = loader.get_class_names()

print(f"Train:   {X_train.shape[0]} samples")
print(f"Val:     {X_val.shape[0]} samples")
print(f"Test:    {X_test.shape[0]} samples")
print(f"Features:{X_train.shape[1]}")
print(f"Classes: {class_names}")

Train:   580800 samples
Val:     11448 samples
Test:    11472 samples
Features:77
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


A continuación se realiza una búsqueda de hiperparámetros con **Randomized Search** para cada modelo baseline de forma individual y, después, se reportan sus métricas en el conjunto de prueba.

## Modelo 1: Regresión Logística Multiclase

In [ ]:
# Baseline 1: Regresión Logística Multiclase + Randomized Search
logistic_param_dist = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "solver": ["lbfgs", "newton-cg"],
    "max_iter": [1000, 2000],
}

logistic_search = RandomizedSearchCV(
    estimator=LogisticRegression(random_state=SEED),
    param_distributions=logistic_param_dist,
    n_iter=1,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)

logistic_search.fit(X_train, y_train)
best_logistic_model = logistic_search.best_estimator_
y_test_pred_logistic = best_logistic_model.predict(X_test)

logistic_test_accuracy = accuracy_score(y_test, y_test_pred_logistic)
logistic_test_f1 = f1_score(y_test, y_test_pred_logistic, average="macro")

print("=== Regresión Logística (Randomized Search + Test) ===")
print(f"Mejores hiperparámetros: {logistic_search.best_params_}")
print(f"Mejor F1-macro CV: {logistic_search.best_score_:.4f}")
print(f"Test Accuracy: {logistic_test_accuracy:.4f}")
print(f"Test F1-macro: {logistic_test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_logistic, target_names=class_names))

cm_logistic = confusion_matrix(y_test, y_test_pred_logistic)
cm_logistic_df = pd.DataFrame(cm_logistic, index=class_names, columns=class_names)
print("Confusion Matrix:")
cm_logistic_df

/home/antonio/Documentos/Universidad/Master Inteligencia Artificial/DISIA/SignCapture/SignCapture-ModelTests/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/antonio/Documentos/Universidad/Master Inteligencia Artificial/DISIA/SignCapture/SignCapture-ModelTests/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT



=== Regresión Logística (Randomized Search + Test) ===
Mejores hiperparámetros: {'solver': 'lbfgs', 'max_iter': 1000}
Mejor F1-macro CV: 0.8840
Test Accuracy: 0.9083
Test F1-macro: 0.9082

Classification Report:
              precision    recall  f1-score   support

           A       0.89      0.92      0.91       478
           B       0.97      0.98      0.98       478
           C       0.88      0.92      0.90       478
           D       0.86      0.81      0.84       478
           E       0.94      0.91      0.93       478
           F       0.89      0.99      0.94       478
           G       0.96      0.94      0.95       478
           H       0.92      0.87      0.90       478
           I       0.98      0.94      0.96       478
           K       0.89      0.94      0.91       478
           L       0.96      0.98      0.97       478
           M       0.73      0.76      0.74       478
           N       0.81      0.76      0.78       478
           O       0.87      0.

/home/antonio/Documentos/Universidad/Master Inteligencia Artificial/DISIA/SignCapture/SignCapture-ModelTests/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,A,B,C,D,E,F,G,H,I,K,...,P,Q,R,S,T,U,V,W,X,Y
A,441,0,0,1,0,3,2,0,0,0,...,0,0,0,9,8,0,0,1,0,0
B,0,470,0,1,0,3,0,0,0,0,...,0,0,0,0,0,0,0,4,0,0
C,0,0,438,8,0,4,1,0,0,0,...,2,10,0,1,0,0,0,0,0,0
D,1,4,13,388,0,5,1,1,2,0,...,4,1,2,0,0,0,0,0,11,1
E,2,4,1,8,437,1,0,1,0,0,...,0,0,0,8,2,0,0,1,6,0
F,0,0,0,3,0,472,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G,9,0,0,0,0,2,448,14,0,2,...,0,0,0,0,1,0,0,0,1,0
H,4,0,3,4,0,3,9,418,0,3,...,0,1,11,3,5,5,1,1,1,3
I,0,4,1,1,5,1,2,0,447,0,...,1,0,1,4,4,0,0,0,0,3
K,0,0,0,1,1,1,2,1,0,449,...,0,0,4,0,0,1,16,0,2,0


## Modelo 2: K-NN

In [17]:
# Baseline 2: K-NN + Randomized Search
knn_param_dist = {
    "n_neighbors": [3, 7, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

knn_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=knn_param_dist,
    n_iter=1,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)

knn_search.fit(X_train, y_train)
best_knn_model = knn_search.best_estimator_
y_test_pred_knn = best_knn_model.predict(X_test)

knn_test_accuracy = accuracy_score(y_test, y_test_pred_knn)
knn_test_f1 = f1_score(y_test, y_test_pred_knn, average="macro")

print("=== K-NN (Randomized Search + Test) ===")
print(f"Mejores hiperparámetros: {knn_search.best_params_}")
print(f"Mejor F1-macro CV: {knn_search.best_score_:.4f}")
print(f"Test Accuracy: {knn_test_accuracy:.4f}")
print(f"Test F1-macro: {knn_test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_knn, target_names=class_names))

cm_knn = confusion_matrix(y_test, y_test_pred_knn)
cm_knn_df = pd.DataFrame(cm_knn, index=class_names, columns=class_names)

print("Confusion Matrix:")
cm_knn_df

=== K-NN (Randomized Search + Test) ===
Mejores hiperparámetros: {'weights': 'uniform', 'n_neighbors': 7, 'metric': 'euclidean'}
Mejor F1-macro CV: 0.9488
Test Accuracy: 0.9568
Test F1-macro: 0.9568

Classification Report:
              precision    recall  f1-score   support

           A       0.93      0.97      0.95       478
           B       0.99      0.99      0.99       478
           C       0.91      0.94      0.92       478
           D       0.96      0.94      0.95       478
           E       0.97      0.96      0.97       478
           F       0.96      0.99      0.97       478
           G       0.95      0.96      0.96       478
           H       0.93      0.92      0.92       478
           I       0.98      0.96      0.97       478
           K       0.94      0.98      0.96       478
           L       0.99      0.98      0.98       478
           M       0.92      0.90      0.91       478
           N       0.93      0.94      0.94       478
           O       0

,A,B,C,D,E,F,G,H,I,K,...,P,Q,R,S,T,U,V,W,X,Y
A,464,0,1,1,1,0,1,1,0,0,...,1,0,0,1,4,0,0,0,0,0
B,0,472,0,1,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C,0,0,448,1,0,4,0,0,1,0,...,1,2,0,0,0,0,0,0,1,0
D,0,0,5,450,0,1,3,0,2,0,...,3,1,0,0,0,0,0,0,3,0
E,1,2,0,0,459,1,0,0,1,0,...,0,0,0,9,0,0,0,0,0,0
F,0,1,2,1,0,471,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G,9,0,1,0,0,1,459,3,0,1,...,1,1,0,0,0,0,0,0,0,0
H,7,0,3,0,0,0,2,438,0,4,...,0,0,7,0,1,10,2,0,0,0
I,2,1,0,3,1,0,3,0,459,0,...,0,1,0,1,1,0,0,0,1,3
K,0,0,0,0,0,0,1,0,0,467,...,0,1,1,0,0,0,7,1,0,0
